# Step 6 — Break the retrieval

*Step 6 of the AI in Industry lab*

---

## Read this before you run anything

You will break the system you just built, in three different ways, and work out which part was actually at fault.

**What you should end up understanding:** Most of the time "the AI is wrong", the retrieval was wrong. This is the most useful debugging instinct in the session.

| | |
|---|---|
| **Cost** | 3 API calls - THE IMPORTANT ONE |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. The k=1 versus k=6 comparison is worth running twice. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 3 API calls.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This is the most important notebook in the lab. Do not skim it — read both answers in the k=1 versus k=6 cells side by side before moving on.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

**This is the most important step in the lab.**

You now have a working RAG system. Time to find out how it fails — because that is the actual job.

In [ ]:
from labcore import corpus, tfidf, grounded

chunks, texts, _ = corpus()
retrieve = tfidf(texts)
ask = grounded(retrieve)      # same retrieve-then-answer as step 5

## (a) Ask something that is genuinely not in there

In [ ]:
print(ask("What are the hostel mess timings?", k=4))

It refuses. **Refusing correctly is a PASSING score**, not a failure.

A system that invents an answer here is worse than useless — it is confidently wrong about your attendance.

## (b) Starve it

Same model, same question. The only thing that changes is `k` — how many chunks we retrieve.

In [ ]:
Q = ("I have 68% attendance in one course because I was away at "
     "placement drives. What happens to me?")

print("########## k=1  (starved) ##########")
print(ask(Q, k=1))

In [ ]:
print("########## k=6  (fed) ##########")
print(ask(Q, k=6))

Read both answers carefully.

- **`k=1`** says you are below 75% and in trouble.
- **`k=6`** says 68% is *within the 10% condonable range*, and placement activity is a listed ground.

**The model was not wrong. It was starved.** At `k=1` it never saw the condonation clause — it answered correctly from half the picture.

> Most of the time "the AI is wrong", the retrieval was wrong.

This is the single most useful debugging instinct in this whole session.

## (c) The right question in the wrong words

In [ ]:
print("regulation wording ->", retrieve("shortage of attendance condoned", k=1)[0][0][:85])
print()
print("student wording    ->", retrieve("exemption for missing too many classes", k=1)[0][0][:85])

The second one retrieves something irrelevant.

**"exemption" and "condoned" share no vocabulary.** TF-IDF matches words, not meaning. The rulebook and the student are describing the same thing in different words, and the search cannot bridge it.

---

## Now change it yourself

Find the value of `k` where the answer flips.

In [ ]:
Q = ("I have 68% attendance in one course because I was away at "
     "placement drives. What happens to me?")

# Try 1, 2, 3, 4... where does it start getting it right?
for k in [1, 2, 3]:
    print(f"########## k={k} ##########")
    print(ask(Q, k=k)[:400])
    print()

# Then try your own question. Ask about something with an exception
# attached to it - those are the ones that need two clauses.

---

### Done with step 6

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.